# Model Testing & Evaluation

This notebook demonstrates the process of testing and evaluating a bioacoustic detection model. It takes a trained model, loads the learned weights from the training phase, and applies it to a new dataset—in this example, recordings containing gibbon calls. The notebook performs predictions on the audio files to detect instances of the target species and then evaluates the model’s performance by comparing the predictions to the annotated ground truth. The goal is to provide a clear, quantitative assessment of how well the model identifies gibbon calls in unseen data, using metrics such as precision, recall, F1-score, and false alarms per hour. This allows us to understand the reliability of the model in practical monitoring scenarios.

In [1]:
try:
    # mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    %cd /content/drive/MyDrive/GithubProject/bioacoustic-detection
except:
    pass


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/GithubProject/bioacoustic-detection


In [ ]:
!pip install yattag
!pip install resampy

In [2]:
import sys
import os

# 1. Define the path to your project folder
project_path = '/content/drive/MyDrive/GithubProject/bioacoustic-detection'

# 2. Add the 'src' directory to sys.path
sys.path.append(os.path.join(project_path, 'src'))

# 3. Move the working directory to your project folder
# This makes sure relative paths for JSON/Audio files work!
os.chdir(project_path)

In [3]:
from testing.prediction_helper import *
from testing.model_evaluator import *

## 1. Model testing

In [4]:
species_folder = '/content/drive/MyDrive/GithubProject/bioacoustic-detection' # Should contain /Audio and /Annotations
lowpass_cutoff = 3500 #2000 # Cutt off for low pass filter
downsample_rate = 8000#4800 # Frequency to downsample to
nyquist_rate = 4000 # Nyquist rate (half of sampling rate)
segment_duration = 4 # how long should a segment be
positive_class = ['gibbon'] # which labels should be bundled together for the positive  class
background_class = ['no-gibbon'] # which labels should be bundled together for the negative  class
file_type = 'svl'
audio_extension = '.wav'
n_fft = 1024 # Hann window length
hop_length = 256 # Sepctrogram hop size
n_mels = 128 # Spectrogram number of mells
f_min = 400 # Spectrogram, minimum frequency for call
f_max = 4000 # Spectrogram, maximum frequency for call
saved_prediction = 'Saved_prediction'
weights_folder = '/content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_weights'

# Define the folder where predictions will be saved
saved_prediction_folder = "/content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_prediction"



In [5]:
mod_test = prediction_helper(
                        species_folder,
                        lowpass_cutoff,
                        downsample_rate,
                        nyquist_rate,
                        segment_duration,
                        positive_class,
                        background_class,
                        n_fft,
                        hop_length,
                        n_mels,
                        f_min,
                        f_max,
                        audio_extension,
                        weights_folder
)

In [6]:
mod_test.predict_all_test_files(weights_folder, saved_prediction_folder)

Processing: HGSM3AC_0+1_20150616_050750
Loading model: /content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_weights/baseline_cnn_20260128_12_best.keras
122/122 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step
Processing: HGSM3AC_0+1_20150618_050600
Loading model: /content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_weights/baseline_cnn_20260128_12_best.keras
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step
Processing: HGSM3AC_0+1_20150622_050600
Loading model: /content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_weights/baseline_cnn_20260128_12_best.keras
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step
Processing: HGSM3AC_0+1_20150717_051400
Loading model: /content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_weights/baseline_cnn_20260128_12_best.keras
68/68 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step
Processing: HGSM3AC_0+1_20150719_051500
Loading model: /content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_weights/baseline_cnn_20260128_12_best.keras
102/102 ━━━━━━━━━━

True

## 2. Evaluation

To evaluate the model’s predictions, we use an event-based detection approach, which compares each predicted call with the annotated ground truth events. A prediction is considered correct if it overlaps with a true event by at least a minimum duration threshold (here, expressed as a percentage of the event’s length). This method is preferred for bioacoustic monitoring because animal calls vary in length, and simple frame-by-frame accuracy may underestimate performance for shorter or longer calls.

We report standard detection metrics:

- `Precision` measures how many of the predicted calls are correct.

- `Recall` measures how many of the actual calls were successfully detected.

- `F1-score` combines precision and recall into a single measure of overall accuracy.

- `False alarms per hour` indicates the rate of incorrect detections, which is important for real-world monitoring to avoid excessive manual verification.

This evaluation strategy gives a balanced and interpretable picture of the model’s ability to detect species calls reliably, regardless of call duration variability.

In [4]:
audio_folder = '/content/drive/MyDrive/GithubProject/bioacoustic-detection/Audio'
annotation_folder = '/content/drive/MyDrive/GithubProject/bioacoustic-detection/Annotations'
prediction_folder = '/content/drive/MyDrive/GithubProject/bioacoustic-detection/Saved_prediction/baseline_cnn_20260128_12_best'

In [5]:
helper = model_evaluator()

df_gt, df_pred = helper.load_all(audio_folder, annotation_folder, prediction_folder)

results = helper.evaluate(
    df_gt=df_gt,
    df_pred=df_pred,
    audio_folder=audio_folder,
    target_label="gibbon",
    min_overlap_pct=0.5   # 50% overlap threshold
)

print("Final evaluation results:")
for k, v in results.items():
    print(f"{k}: {v}")

 EVENT-BASED DETECTION EVALUATION (percentage overlap) 
Target species        : gibbon
True events (GT)      : 463
True Positives (TP)   : 456
False Negatives (FN)  : 7
False Positives (FP)  : 93
---------------------------------------------
Precision             : 0.831
Recall                : 0.985
F1-score              : 0.901
False alarms / hour   : 9.05
Final evaluation results:
TP: 456
FP: 93
FN: 7
Precision: 0.8306010928961749
Recall: 0.9848812095032398
F1: 0.9011857707509882
FP_per_hour: 9.05302019714024


The model performed very well at detecting gibbon calls in the audio recordings. Out of 463 true gibbon events, it correctly identified 456, giving a recall of 98.5%, which means it missed very few calls. The precision was 83%, meaning that about 17% of the detected events were actually false alarms. Overall, the F1-score of 90% shows a strong balance between detecting most calls and avoiding false positives. On average, the model produced about 9 false alarms per hour of audio, which is a manageable rate for monitoring purposes. In simple terms, the model reliably finds gibbon calls while making relatively few mistakes.